# 1、流式调用（stream）

举例：

In [ ]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os

# 从.env文件中加载环境变量
load_dotenv(override=True)

DASHSCOPE_API_KEY = os.getenv("DASHSCOPE_API_KEY")
DASHSCOPE_BASE_URL = os.getenv("DASHSCOPE_BASE_URL")

model = init_chat_model(
    model="qwen3.7-max",
    model_provider="openai",
    # temperature=1.5,
    api_key=DASHSCOPE_API_KEY,
    base_url=DASHSCOPE_BASE_URL,
    # max_tokens=10,
)

In [ ]:
for chunk in model.stream("帮我简单解释一下，什么是人工智能？"):
    print(chunk.text, end="", flush=True)

# 2、批量调用（batch）

举例1：一次性接收所有的响应

In [ ]:
messages = [
    "你好，你是谁？"
    "2 + 3 * 5 = ？"
    "中国的首都在哪里？"
]

responses = model.batch(messages)

for response in responses:
    print(response.text, end="", flush=True)

举例2：按照完成的顺序接收响应

In [ ]:
messages = [
    "你好，你是谁？"
    "2 + 3 * 5 = ？"
    "中国的首都在哪里？"
]

responses = model.batch_as_completed(messages)

for response in responses:
    print(response)

举例3：性能对比

使用batch

In [21]:
# 准备多个输入
inputs = [
    "翻译成英文：春天来了",
    "翻译成英文：夏天很热",
    "翻译成英文：秋天落叶",
    "翻译成英文：冬天下雪"
]

# √ 批量调用（高效）
import time

start = time.time()
responses = model.batch(inputs)
batch_time = time.time() - start

print("批量调用结果：")
for i, response in enumerate(responses):
    print(f"{i + 1}. {response.text}", end="", flush=True)
print("\n")
print(f"耗时：{batch_time:.2f}s\n")


批量调用结果：
1. Spring is here.2. It is very hot in summer. 

（或者更口语化的表达：It's really hot in the summer.）3. Autumn Leaves4. It snows in winter.

耗时：8.76s



对比，演示循环调用invoke

In [22]:
# 准备多个输入 × 循环调用（低效，仅用于对比）
inputs = [
    "翻译成英文：春天来了",
    "翻译成英文：夏天很热",
    "翻译成英文：秋天落叶",
    "翻译成英文：冬天下雪"
]

start = time.time()
loop_responses = []
for inp in inputs:
    response = model.invoke(inp)
    loop_responses.append(response)
loop_time = time.time() - start

for i, response in enumerate(loop_responses):
    print(f"{i + 1}. {response.content}")

print(f"循环调用耗时：{loop_time:.2f}s\n")

print(f"批量调用节省：{((loop_time - batch_time) / loop_time * 100):.2f}%")

1. Spring is here.
2. It's very hot in summer.
3. Fallen Leaves in Autumn
4. “冬天下雪”翻译成英文最常用、最自然的表达是：

**It snows in winter.** 

（或者也可以说 **It snows in the winter.**）

如果你需要其他语境下的表达，还可以参考以下几种：
* **Snow falls in winter.** （偏书面或文学表达：冬雪飘落）
* **Winter brings snow.** （偏文学表达：冬天带来了雪）
* **Snowy winter** （作为名词短语：下雪的冬天）
循环调用好事：35.54s

批量调用节省：75.34%
